# Unified Multi-Server MCP Agent in LangChain: AWS Documentation + Browser MCP

This notebook demonstrates how to integrate **two official Model Context Protocol (MCP) servers** directly into a single **LangChain Agent** (`create_agent`) using `MultiServerMCPClient` and persistent **SQLite memory** (`AsyncSqliteSaver`):

1. **[AWS Documentation MCP Server](https://awslabs.github.io/mcp/servers/aws-documentation-mcp-server)** (`awslabs.aws-documentation-mcp-server@latest` via `uvx`)
   - **Transport**: `stdio`
   - **Capabilities**: Searches official AWS documentation, reads and converts AWS documentation pages or specific sections into Markdown, searches large documentation tables, and recommends related AWS docs.
   - **Tools Exposed (5)**: `search_documentation`, `read_documentation`, `read_sections`, `search_table`, `recommend`.

2. **[Browser MCP Server](https://docs.browsermcp.io/setup-server)** (`@browsermcp/mcp@latest` via `npx`)
   - **Transport**: `stdio` (LangChain MCP Client $\leftrightarrow$ Browser MCP Server) + Local WebSocket `ws://localhost:9009` (Browser MCP Server $\leftrightarrow$ Browser MCP Chrome Extension).
   - **Prerequisite**: Install the [Browser MCP Chrome Extension](https://docs.browsermcp.io/setup-extension), open a Chrome tab, click the extension icon, and click **"Connect"**.
   - **Capabilities**: Automates your live Chrome browser tab—navigating to URLs (`amazon.in`, `flipkart.com`), capturing page accessibility snapshots (`[ref=...]`), typing into search boxes, clicking buttons, and extracting live product prices.
   - **Tools Exposed (12)**: `browser_navigate`, `browser_snapshot`, `browser_type`, `browser_click`, `browser_hover`, `browser_select_option`, `browser_press_key`, `browser_wait`, `browser_go_back`, `browser_go_forward`, `browser_get_console_logs`, `browser_screenshot`.

---

### System Architecture & Intent Routing Flow

```text
                         ┌──────────────────────────────────────────┐
                         │               User Query                 │
                         │  • "Explain AWS Lambda concurrency..."   │
                         │  • "Compare iPhone 17 price on Amazon    │
                         │     and Flipkart..."                     │
                         └────────────────────┬─────────────────────┘
                                              │
                                              ▼
                         ┌──────────────────────────────────────────┐
                         │      LangChain Agent (create_agent)      │
                         │   Model: Gemini 2.5 Flash + System Rules │◄───► AsyncSqliteSaver
                         │   Intent Classifier & Tool Orchestrator  │      (Persistent DB)
                         └────────────────────┬─────────────────────┘
                                              │
                         ┌────────────────────┴─────────────────────┐
                         │   MultiServerMCPClient (17 MCP Tools)    │
                         └─────────┬──────────────────────┬─────────┘
                                   │                      │
          [Intent: AWS Cloud Docs] │                      │ [Intent: Browser Automation]
                                   ▼                      ▼
     ┌──────────────────────────────────────┐   ┌──────────────────────────────────────────┐
     │  Server 1: aws_docs (stdio / uvx)    │   │  Server 2: browsermcp (stdio / npx)      │
     │  awslabs.aws-documentation-mcp-server│   │  @browsermcp/mcp@latest                  │
     ├──────────────────────────────────────┤   ├──────────────────────────────────────────┤
     │ • search_documentation               │   │ • browser_navigate   • browser_type      │
     │ • read_documentation                 │   │ • browser_snapshot   • browser_click     │
     │ • read_sections                      │   │ • browser_press_key  • browser_wait      │
     │ • search_table                       │   │ • browser_go_back    • browser_screenshot│
     │ • recommend                          │   └────────────────────┬─────────────────────┘
     └──────────────────┬───────────────────┘                        │ WebSocket (ws://localhost:9009)
                        │                                            ▼
                        ▼                               ┌──────────────────────────────────┐
          Official AWS Docs Search &                    │  Browser MCP Chrome Extension    │
            Markdown Page Fetcher                       │  Connected Chrome Tab:           │
            (docs.aws.amazon.com)                       │  Amazon.in & Flipkart.com        │
                                                        └──────────────────────────────────┘
```

## 1. Environment Setup & Prerequisite Checks

First, we load our environment variables (`.env`) for `GOOGLE_API_KEY` / `GEMINI_API_KEY` and locate the `uvx` (for AWS Documentation MCP Server) and `npx` (for Browser MCP Server) executables.

In [1]:
import asyncio
import glob
import json
import logging
import os
import re
import shutil
import warnings
from contextlib import AsyncExitStack
from typing import Any
from dotenv import load_dotenv

# Suppress JSON schema warnings for additionalProperties
warnings.filterwarnings("ignore", message=".*additionalProperties.*")
logging.getLogger("google").setLevel(logging.ERROR)
logging.getLogger("langchain_google_genai").setLevel(logging.ERROR)

# Search up directory tree to find .env file
def load_env_robust():
    path = os.getcwd()
    for _ in range(5):
        dotenv_path = os.path.join(path, ".env")
        if os.path.exists(dotenv_path):
            load_dotenv(dotenv_path=dotenv_path)
            print(f"Loaded .env from: {dotenv_path}")
            return
        path = os.path.dirname(path)
    load_dotenv(dotenv_path="../.env")

load_env_robust()

# Ensure GEMINI_API_KEY is mapped from GOOGLE_API_KEY
if "GOOGLE_API_KEY" in os.environ and "GEMINI_API_KEY" not in os.environ:
    os.environ["GEMINI_API_KEY"] = os.environ["GOOGLE_API_KEY"]

# Locate uvx (for AWS Documentation MCP) and npx (for Browser MCP)
UVX_PATH = (
    shutil.which("uvx")
    or os.path.expanduser("~/.local/bin/uvx")
    or os.path.expanduser("~/.cargo/bin/uvx")
    or "/opt/homebrew/bin/uvx"
)
NPX_PATH = shutil.which("npx") or "/opt/homebrew/bin/npx" or "/usr/local/bin/npx"

print(f"GEMINI_API_KEY : {'Present' if 'GEMINI_API_KEY' in os.environ else 'Missing'}")
print(f"uvx binary     : {UVX_PATH} (exists={os.path.exists(UVX_PATH)})")
print(f"npx binary     : {NPX_PATH} (exists={os.path.exists(NPX_PATH)})")

Loaded .env from: /Users/sachinmishra/Desktop/Agents_From_Scratch/.env
GEMINI_API_KEY : Present
uvx binary     : /Users/sachinmishra/.langflow/uv/uvx (exists=True)
npx binary     : /opt/homebrew/bin/npx (exists=True)


## 2. Connect Both MCP Servers via `MultiServerMCPClient`

We configure `MultiServerMCPClient` with the exact server specifications from both official documentation links:

1. **`aws_docs`** ([AWS Documentation MCP Server](https://awslabs.github.io/mcp/servers/aws-documentation-mcp-server)):
   ```json
   {
     "command": "uvx",
     "args": ["awslabs.aws-documentation-mcp-server@latest"],
     "env": {"FASTMCP_LOG_LEVEL": "ERROR", "AWS_DOCUMENTATION_PARTITION": "aws"}
   }
   ```
2. **`browsermcp`** ([Browser MCP Server](https://docs.browsermcp.io/setup-server)):
   ```json
   {
     "command": "npx",
     "args": ["@browsermcp/mcp@latest"]
   }
   ```

> **Why Persistent Sessions (`mcp_client.session(...)`) Matter for Browser MCP**:
> By opening persistent `stdio` sessions via `AsyncExitStack` and `load_mcp_tools(session)`, the `@browsermcp/mcp` server stays alive in the background and keeps listening on `ws://localhost:9009`. This allows you to open Chrome, click the **Browser MCP Extension**, and click **"Connect"** so all browser tool calls execute on your connected browser tab.

In [2]:
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_mcp_adapters.tools import load_mcp_tools

# 1. Initialize MultiServerMCPClient with only the two official MCP servers
mcp_client = MultiServerMCPClient(
    {
        "aws_docs": {
            "command": UVX_PATH,
            "args": ["awslabs.aws-documentation-mcp-server@latest"],
            "transport": "stdio",
            "env": {
                **os.environ,
                "FASTMCP_LOG_LEVEL": "ERROR",
                "AWS_DOCUMENTATION_PARTITION": "aws",
            },
        },
        "browsermcp": {
            "command": NPX_PATH,
            "args": ["-y", "@browsermcp/mcp@latest"],
            "transport": "stdio",
        },
    }
)

# 2. Open persistent MCP sessions so @browsermcp/mcp keeps ws://localhost:9009 open
mcp_exit_stack = AsyncExitStack()
aws_session = await mcp_exit_stack.enter_async_context(mcp_client.session("aws_docs"))
browser_session = await mcp_exit_stack.enter_async_context(mcp_client.session("browsermcp"))

# 3. Load LangChain tools from both active MCP sessions
aws_tools = await load_mcp_tools(aws_session, server_name="aws_docs")
browser_tools = await load_mcp_tools(browser_session, server_name="browsermcp")

# Enable graceful tool error reporting so MCP tool errors are returned to the LLM as text
for t in aws_tools + browser_tools:
    t.handle_tool_error = True

all_mcp_tools = aws_tools + browser_tools

print(f"=== Loaded {len(aws_tools)} Tools from AWS Documentation MCP Server (`aws_docs`) ===")
for t in aws_tools:
    first_line = t.description.strip().splitlines()[0] if t.description else ""
    print(f"  • {t.name:<24} | Args: {list(t.args.keys())} | {first_line}")

print(f"\n=== Loaded {len(browser_tools)} Tools from Browser MCP Server (`browsermcp`) ===")
for t in browser_tools:
    first_line = t.description.strip().splitlines()[0] if t.description else ""
    print(f"  • {t.name:<24} | Args: {list(t.args.keys())} | {first_line}")

print(f"\nTotal Combined MCP Tools Ready for LangChain Agent: {len(all_mcp_tools)}")

=== Loaded 5 Tools from AWS Documentation MCP Server (`aws_docs`) ===
  • read_documentation       | Args: ['url', 'max_length', 'start_index'] | Fetch and convert an AWS documentation page to markdown format.
  • read_sections            | Args: ['url', 'section_titles'] | Extract specific sections from AWS documentation pages by title.
  • search_table             | Args: ['url', 'section_title', 'query', 'max_rows'] | Search for specific rows in a large documentation table.
  • search_documentation     | Args: ['search_phrase', 'search_intent', 'limit', 'product_types', 'guide_types'] | Search AWS documentation using the official AWS Documentation Search API.
  • recommend                | Args: ['url'] | Get content recommendations for an AWS documentation page.

=== Loaded 12 Tools from Browser MCP Server (`browsermcp`) ===
  • browser_navigate         | Args: ['url'] | Navigate to a URL
  • browser_go_back          | Args: [] | Go back to the previous page
  • browser_go_forward 

## 3. Build the LangChain Agent with SQLite Checkpointer (`AsyncSqliteSaver`)

We bind all **17 MCP tools** (`5` AWS Documentation tools + `12` Browser Automation tools) to `ChatGoogleGenerativeAI(model="gemini-2.5-flash")` using LangChain's `create_agent`, backed by `AsyncSqliteSaver` (`aws_browser_mcp_history.db`) for multi-turn conversation persistence.

In [ ]:
import aiosqlite
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.checkpoint.sqlite.aio import AsyncSqliteSaver

# Initialize persistent SQLite checkpointer for multi-turn agent memory
sqlite_conn = await aiosqlite.connect("aws_browser_mcp_history.db")
memory = AsyncSqliteSaver(sqlite_conn)

# Initialize Gemini 3.1 Flash Lite model
llm = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite", temperature=0)

SYSTEM_PROMPT = (
    "You are an intelligent Multi-Server MCP Agent equipped with two official Model Context Protocol (MCP) servers:\n"
    "1. **AWS Documentation MCP Server (`aws_docs`)**: Use `search_documentation`, `read_documentation`, "
    "`read_sections`, `search_table`, and `recommend` whenever the user asks about AWS services, cloud architecture, "
    "quotas, pricing, or configurations.\n"
    "2. **Browser MCP Server (`browsermcp`)**: Use `browser_navigate`, `browser_snapshot`, `browser_type`, "
    "`browser_click`, `browser_press_key`, and `browser_wait` whenever the user asks to automate the browser "
    "or check live product prices on e-commerce websites like Amazon.in and Flipkart.com.\n\n"
    "CRITICAL ROUTING & FORMATTING RULES:\n"
    "- **For AWS Documentation Queries**: First call `search_documentation` to find official AWS doc pages, then call "
    "`read_documentation` or `read_sections` on the most relevant URL to retrieve authoritative details. Always cite "
    "the official `https://docs.aws.amazon.com/...` URLs in your final answer.\n"
    "- **For Browser Automation / Price Comparison Queries (e.g., iPhone 17 on Amazon vs Flipkart)**:\n"
    "  1. Call `browser_navigate` to open `https://www.amazon.in`, then call `browser_snapshot` to inspect elements, "
    "and use `browser_type` (with `submit=True`) or `browser_click` to search for the product.\n"
    "  2. Call `browser_navigate` to open `https://www.flipkart.com`, then call `browser_snapshot` to inspect elements, "
    "and use `browser_type` (with `submit=True`) or `browser_click` to search for the product.\n"
    "  3. If the Browser MCP server reports that the Chrome Extension is not connected yet, do not loop indefinitely—report "
    "the exact Browser MCP tool calls you attempted and remind the user to click 'Connect' in the Browser MCP Chrome Extension.\n"
    "- **Compact Markdown Tables**: When generating Markdown tables, NEVER pad cells with extra spaces. Use compact syntax like `| col1 | col2 |`."
).strip()

mcp_agent = create_agent(
    model=llm,
    tools=all_mcp_tools,
    system_prompt=SYSTEM_PROMPT,
    checkpointer=memory,
)

def extract_text(content: Any) -> str:
    """Extracts clean text from string or list-of-blocks AIMessage content."""
    if isinstance(content, list):
        content = "\n".join(
            block.get("text", "") for block in content if isinstance(block, dict) and block.get("type") == "text"
        )
    text = str(content)
    text = re.sub(r"-{8,}", "--------", text)
    return re.sub(r" {4,}", " ", text)

print("Unified LangChain Multi-Server MCP Agent created with SQLite persistence!")

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


Unified LangChain Multi-Server MCP Agent created with SQLite persistence!


## 4. Use Case 1: Querying the AWS Documentation MCP Server

Let's test our agent with an **AWS Documentation** question. Notice how the agent automatically selects the `aws_docs` MCP tools (`search_documentation` $\rightarrow$ `read_sections` / `read_documentation`) to query official AWS documentation in real time.

In [4]:
aws_query = (
    "What are the AWS Lambda concurrency quotas, and what is the exact difference between "
    "Reserved Concurrency and Provisioned Concurrency? Search the official AWS documentation "
    "and include the official documentation links."
)

print(f"User Query: {aws_query}\n" + "=" * 90)

config_aws = {"configurable": {"thread_id": "aws_docs_session_pure_1"}}
response_aws = await mcp_agent.ainvoke(
    {"messages": [HumanMessage(content=aws_query)]},
    config=config_aws,
)

print("\n--- MCP Tool Calls Executed ---")
for msg in response_aws["messages"]:
    if hasattr(msg, "tool_calls") and msg.tool_calls:
        for tc in msg.tool_calls:
            print(f"  [MCP Tool] {tc['name']}({json.dumps(tc['args'])})")

print("\n--- Final Agent Response ---\n")
print(extract_text(response_aws["messages"][-1].content))

User Query: What are the AWS Lambda concurrency quotas, and what is the exact difference between Reserved Concurrency and Provisioned Concurrency? Search the official AWS documentation and include the official documentation links.


CancelledError: 

## 5. Use Case 2: Browser Automation via Browser MCP Server (`@browsermcp/mcp`)

Now let's ask the agent to find and compare the **price of iPhone 17 on Amazon (`amazon.in`) and Flipkart (`flipkart.com`)** using the **Browser MCP Server**.

### Before Running This Cell in Interactive Browser Mode:
1. Install the **[Browser MCP Chrome Extension](https://docs.browsermcp.io/setup-extension)** in Google Chrome.
2. Open a tab in Chrome, click the **Browser MCP** extension icon in the toolbar, and click **"Connect"** (it connects to the persistent `browsermcp` session started in Cell 2 on `ws://localhost:9009`).
3. Run the cell below—the agent will call `browser_navigate`, `browser_snapshot`, and `browser_type` directly on your connected Chrome tab to search `amazon.in` and `flipkart.com` for **iPhone 17** and compare the prices.

In [5]:
browser_query = (
    "Using the browser automation tools, go to Amazon (https://www.amazon.in) and Flipkart (https://www.flipkart.com), "
    "search for 'iPhone 17' on both websites, and compare the prices of iPhone 17 across both platforms."
)

print(f"User Query: {browser_query}\n" + "=" * 90)

config_shopping = {"configurable": {"thread_id": "browser_mcp_iphone17_pure_1"}}
response_browser = await mcp_agent.ainvoke(
    {"messages": [HumanMessage(content=browser_query)]},
    config=config_shopping,
)

print("\n--- Browser MCP Tool Calls Executed ---")
for msg in response_browser["messages"]:
    if hasattr(msg, "tool_calls") and msg.tool_calls:
        for tc in msg.tool_calls:
            print(f"  [Browser MCP Tool] {tc['name']}({json.dumps(tc['args'])})")

print("\n--- Final Agent Response ---\n")
print(extract_text(response_browser["messages"][-1].content))

User Query: Using the browser automation tools, go to Amazon (https://www.amazon.in) and Flipkart (https://www.flipkart.com), search for 'iPhone 17' on both websites, and compare the prices of iPhone 17 across both platforms.


ClosedResourceError: 

## 6. Clean Up Persistent MCP Sessions & SQLite Connection

When you are finished automating your browser and querying AWS documentation, run the cell below to cleanly close the SQLite database connection and the MCP sessions.

In [ ]:
# Close SQLite connection (keep mcp_exit_stack open while actively using the Chrome Extension)
await sqlite_conn.close()
await mcp_exit_stack.aclose()
print("SQLite connection and MCP server sessions closed cleanly.")

SQLite connection and MCP server sessions closed cleanly.
